In [ ]:
%%sql -r dataframe_1
USE DATABASE WEATHER;
SHOW TABLES;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE WEATHER_TABLE (
    DATA VARIANT,
    INGESTED_TS TIMESTAMP DEFAULT sysdate()
);

In [ ]:
%%sql -r dataframe_12
select * from WEATHER_TABLE;

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE STREAM WEATHER_STREAM ON TABLE WEATHER_TABLE;

In [ ]:
%%sql -r dataframe_19
SELECT * FROM WEATHER_STREAM

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TABLE WEATHER_CLEAN AS
SELECT
    DATA:main.temp::FLOAT AS TEMP,
    DATA:main.temp_min::FLOAT AS TEMP_MIN,
    DATA:main.temp_max::FLOAT AS TEMP_MAX,
    DATA:main.pressure::FLOAT AS PRESSURE,
    DATA:main.humidity::FLOAT AS HUMIDITY,
    TO_TIMESTAMP(DATA:main.dt::STRING, 'yyyy-mm-dd hh24:mi:ss') AS RECORD_TS,
    INGESTED_TS AS INGESTED_TS
FROM WEATHER_STREAM
WHERE 1=0;

In [ ]:
CREATE OR REPLACE TASK WEATHER_JSON_PARSE_TASK
WAREHOUSE = COMPUTE_WH
SCHEDULE = '1 MINUTE'
AS
INSERT INTO WEATHER_CLEAN(TEMP, TEMP_MIN, TEMP_MAX, PRESSURE, HUMIDITY, RECORD_TS, INGESTED_TS)
SELECT
    DATA:main.temp::FLOAT,
    DATA:main.temp_min::FLOAT,
    DATA:main.temp_max::FLOAT,
    DATA:main.pressure::FLOAT,
    DATA:main.humidity::FLOAT,
    TO_TIMESTAMP(SUBSTR(DATA:main.dt,0,19), 'yyyy-mm-dd hh24:mi:ss'),
    INGESTED_TS
FROM WEATHER_STREAM;

ALTER TASK WEATHER_JSON_PARSE_TASK RESUME; 

In [ ]:
%%sql -r dataframe_11
select * from WEATHER_CLEAN

In [ ]:
%%sql -r dataframe_14
SHOW TASKS;
--ALTER TASK WEATHER_JSON_PARSE_TASK RESUME;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE STREAM WEATHER_CLEAN_STREAM ON TABLE WEATHER_CLEAN;

In [ ]:
select * from WEATHER_CLEAN_STREAM;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE WEATHER_AGG AS
SELECT
    TEMP,
    TEMP_MIN,
    TEMP_MAX,
    PRESSURE,
    HUMIDITY,
    RECORD_TS,
    INGESTED_TS
FROM WEATHER_CLEAN
WHERE 1=0;

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE TASK WEATHER_HOURLY_AGG_TASK
WAREHOUSE = 'COMPUTE_WH'
SCHEDULE = 'USING CRON 59 * * * * UTC'
AS

MERGE INTO WEATHER_AGG A USING 
(SELECT
    ROUND(AVG(TEMP),2) TEMP,
    MIN(TEMP_MIN) TEMP_MIN,
    MAX(TEMP_MAX) TEMP_MAX,
    ROUND(AVG(PRESSURE),2) PRESSURE,
    ROUND(AVG(HUMIDITY),2) HUMIDITY,
    TO_TIMESTAMP(SUBSTR(MAX(RECORD_TS),0,13), 'yyyy-mm-dd hh24') RECORD_TS,
    SYSDATE() INGESTED_TS
FROM WEATHER_CLEAN_STREAM) B ON A.RECORD_TS = B.RECORD_TS

WHEN MATCHED THEN UPDATE SET
A.TEMP = B.TEMP,
A.TEMP_MIN = B.TEMP_MIN,
A.TEMP_MAX = B.TEMP_MAX,
A.PRESSURE = B.PRESSURE,
A.HUMIDITY = B.HUMIDITY,
A.INGESTED_TS = B.INGESTED_TS

WHEN NOT MATCHED THEN INSERT (
    TEMP,
    TEMP_MIN,
    TEMP_MAX,
    PRESSURE,
    HUMIDITY,
    RECORD_TS,
    INGESTED_TS
)
VALUES (
    B.TEMP,
    B.TEMP_MIN,
    B.TEMP_MAX,
    B.PRESSURE,
    B.HUMIDITY,
    B.RECORD_TS,
    B.INGESTED_TS
);

ALTER TASK WEATHER_HOURLY_AGG_TASK RESUME; 

In [ ]:
%%sql -r dataframe_16
EXECUTE TASK WEATHER_HOURLY_AGG_TASK

In [ ]:
%%sql -r dataframe_17
SELECT * FROM WEATHER_AGG